In [7]:
import json

with open("artists_songs_data_full.jsonl", "r") as f:
    data = json.load(f)

print(len(data))       # number of items

# print(json.dumps(data, indent=2))

499


In [15]:
from datetime import datetime

def parse_year_safe(date_str):
    if not date_str or not isinstance(date_str, str):
        return None
    y = date_str.split("-")[0]
    if not (y.isdigit() and len(y) == 4):
        return None
    year = int(y)
    current_year = datetime.now().year
    # ignore implausible or future years
    if year < 1950 or year > current_year:
        return None
    return year

def extract_features(artist):
    works = artist.get("works", [])
    num_works = len(works)
    if num_works == 0:
        return None

    artist_mbid = artist.get("mbid")
    collab_counts = []
    unique_collabs = set()
    valid_years = []

    for w in works:
        if not isinstance(w, dict):
            continue

        # Collaborators (excluding the artist themself)
        collabs = [c for c in w.get("collaborators", []) if c.get("mbid") != artist_mbid]
        collab_counts.append(len(collabs))
        for c in collabs:
            if c.get("mbid"):
                unique_collabs.add(c["mbid"])

        # Release year
        y = parse_year_safe(w.get("release_date"))
        if y is not None:
            valid_years.append(y)

    avg_collaborators = sum(collab_counts) / num_works if num_works else 0.0
    avg_diversity = len(unique_collabs) / num_works if num_works else 0.0
    num_active_years = (
        max(valid_years) - min(valid_years) + 1 if len(valid_years) > 1 else 1
    ) if valid_years else 0

    return {
        "artist_mbid": artist_mbid,
        "artist_name": artist.get("artist_name"),
        "avg_collaborators_per_song": avg_collaborators,
        "num_works": num_works,
        "num_active_years": num_active_years,
        "avg_diversity": avg_diversity,
    }


# Process all artists
records = [extract_features(a) for a in data if extract_features(a) is not None]
df = pd.DataFrame(records)

print(df)

                              artist_mbid         artist_name  \
0    c8b03190-306c-4120-bb0b-6f2ebfc06ea9          The Weeknd   
1    f6beac20-5dfe-4d1f-ae02-0b0a740aafd6  Tyler, The Creator   
2    20244d07-534f-4eff-b4d4-930878889970        Taylor Swift   
3    a74b1b7f-71a5-4011-9441-d0b5e4122711           Radiohead   
4    381086ea-f511-4aba-bdf9-71c753dc5077      Kendrick Lamar   
..                                    ...                 ...   
460  fee6a7fb-80d9-4290-a171-e48f1f20e381           Tom Odell   
461  923e649b-21d3-4b45-b109-50e4978002b6              Aurora   
462  94c338ff-1985-4429-9dc8-997b61bb5932               B.o.B   
463  20634ec0-4bdb-47dc-9bf4-07ed76f8154a        Montell Fish   
464  d2582c77-6b0f-4b2f-802f-8f3cad1a5fec               Jimin   

     avg_collaborators_per_song  num_works  num_active_years  avg_diversity  
0                      3.681818        198                42       1.500000  
1                      1.024845        161                60   

In [9]:
df["num_active_years"]

0      42
1      60
2      98
3      88
4      40
       ..
460    57
461    14
462    39
463     3
464     5
Name: num_active_years, Length: 465, dtype: int64

In [10]:
def parse_year_safe(date_str):
    if not date_str or not isinstance(date_str, str):
        return None
    y = date_str.split("-")[0]
    return int(y) if y.isdigit() and len(y) == 4 else None

def print_artist_years(data, artist_name="Taylor Swift"):
    for artist in data:
        if artist.get("artist_name", "").lower() == artist_name.lower():
            print(f"Years for {artist['artist_name']}:")
            years = []
            for w in artist.get("works", []):
                y = parse_year_safe(w.get("release_date"))
                if y:
                    years.append(y)
            years = sorted(set(years))
            print(years)
            print(f"Total valid works: {len(artist['works'])}")
            return
    print(f"No artist found with name '{artist_name}'")

# Example usage
print_artist_years(data, "Taylor Swift")


Years for Taylor Swift:
[1928, 1942, 1953, 1964, 1969, 1976, 1977, 1978, 1981, 1983, 1984, 1987, 1989, 1992, 1995, 1996, 1998, 2000, 2001, 2002, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Total valid works: 343


In [14]:
def find_bad_dates(data, artist_name="Taylor Swift"):
    if not isinstance(data, list):
        print("Data should be a list of artist dictionaries.")
        return

    for artist in data:
        # Ensure this entry is a dict and has an artist name
        if not isinstance(artist, dict):
            continue
        name = artist.get("artist_name")
        if not isinstance(name, str):
            continue  # skip invalid or missing artist_name
        
        # Compare case-insensitively
        if name.lower() == artist_name.lower():
            for w in artist.get("works", []) or []:
                release_date = w.get("release_date") if isinstance(w, dict) else None
                if isinstance(release_date, str) and release_date.startswith("1928"):
                    print("Bad date found:", w.get("name"), release_date)

find_bad_dates(data)

Bad date found: Silent Night 1928-01-01


In [17]:
import pandas as pd

# Read your features DataFrame (already built from JSON)
# df = pd.DataFrame(records)

# Read the top 500 artists CSV
top500 = pd.read_csv("artists_top_500.csv")

# Inspect the CSV structure (optional)
# print(top500.head())

# Merge on mbid columns
merged = df.merge(
    top500,
    how="left",               # keep all artists from your features df
    left_on="artist_mbid",
    right_on="mbid"
)

# Drop the duplicate mbid column if you like
merged = merged.drop(columns=["mbid"])

# Rename for clarity (optional)
merged = merged.rename(columns={
    "listeners": "lastfm_listeners",
    "playcount": "lastfm_playcount",
    "rank": "lastfm_rank"
})

merged


,artist_mbid,artist_name,avg_collaborators_per_song,num_works,num_active_years,avg_diversity,lastfm_rank,name,lastfm_listeners,lastfm_playcount
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,3.681818,198,42,1.500000,1,The Weeknd,5022809,1021358879
1,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",1.024845,161,60,0.714286,2,"Tyler, The Creator",4069112,961722659
2,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,1.259475,343,73,0.489796,3,Taylor Swift,5654543,3405301973
3,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,3.547170,212,63,0.306604,4,Radiohead,7907062,1266279394
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,3.481651,218,40,1.857798,5,Kendrick Lamar,4832257,954255600
...,...,...,...,...,...,...,...,...,...,...
460,fee6a7fb-80d9-4290-a171-e48f1f20e381,Tom Odell,0.744681,47,57,0.617021,495,Tom Odell,1918675,46147775
461,923e649b-21d3-4b45-b109-50e4978002b6,Aurora,1.846154,13,14,1.384615,496,Aurora,1701389,82613857
462,94c338ff-1985-4429-9dc8-997b61bb5932,B.o.B,3.490196,51,39,2.686275,498,B.o.B,3013828,43965333
463,20634ec0-4bdb-47dc-9bf4-07ed76f8154a,Montell Fish,1.000000,5,3,0.200000,499,Montell Fish,1243810,39699348


In [18]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

cols_to_scale = ["avg_collaborators_per_song", "num_works", "num_active_years",
                 "avg_diversity", "lastfm_listeners", "lastfm_playcount"]

merged[cols_to_scale] = scaler.fit_transform(merged[cols_to_scale])

In [21]:
merged.to_csv("artist_features_ready.csv", index=False)
df = merged

In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# --- inputs ---
feature_cols = [
    "avg_collaborators_per_song",
    "num_works",
    "num_active_years",
    "avg_diversity",
]
target_col = "lastfm_listeners"      # <-- or "lastfm_playcount"

# Basic hygiene
use = df.dropna(subset=feature_cols + [target_col]).copy()

X = use[feature_cols].values
y = use[target_col].values.reshape(-1, 1)

# Pipeline: scale features, MLP with early stopping
base = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42
    ))
])

# Wrap with log1p/expm1 transform on the target
reg = TransformedTargetRegressor(
    regressor=base,
    func=np.log1p,    # y -> log(1+y) for training
    inverse_func=np.expm1
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y.ravel(), test_size=0.2, random_state=42
)

reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R^2:", r2_score(y_test, pred))


MAE: 0.09790297424321724
R^2: 0.30909529049009943


In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

feature_cols = [
    "avg_collaborators_per_song",
    "num_works",
    "num_active_years",
    "avg_diversity",
]
target_cols = ["lastfm_listeners", "lastfm_playcount"]

use = df.dropna(subset=feature_cols + target_cols).copy()

X = use[feature_cols].values
Y = use[target_cols].values   # shape (n, 2)

# We’ll log1p the targets manually for multioutput and invert after
Y_log = np.log1p(Y)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y_log, test_size=0.2, random_state=42
)

base = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42
    ))
])

multi = MultiOutputRegressor(base)
multi.fit(X_train, Y_train)

Y_pred_log = multi.predict(X_test)
Y_pred = np.expm1(Y_pred_log)
Y_true = np.expm1(Y_test)

mae_listeners = mean_absolute_error(Y_true[:, 0], Y_pred[:, 0])
mae_playcount = mean_absolute_error(Y_true[:, 1], Y_pred[:, 1])
r2_listeners = r2_score(Y_true[:, 0], Y_pred[:, 0])
r2_playcount = r2_score(Y_true[:, 1], Y_pred[:, 1])

print(f"Listeners  -> MAE: {mae_listeners:.2f}, R^2: {r2_listeners:.3f}")
print(f"Playcount  -> MAE: {mae_playcount:.2f}, R^2: {r2_playcount:.3f}")


Listeners  -> MAE: 0.10, R^2: 0.309
Playcount  -> MAE: 0.06, R^2: -0.124


In [24]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    random_state=42
)

In [27]:
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print("R²:", r2_score(y_test, pred))


R²: 0.3207588760529342
